# PartnerLens Data Inventory and Validation

## Final Capstone Submission — Phase 8

This notebook demonstrates:

- repository-relative data loading;
- data inventory and schema inspection;
- data quality checks;
- preparation of processed PartnerLens datasets;
- cross-table validation;
- final Phase 8 validation using the reusable `src.data_preparation` module.

All project data is synthetic and intended only for academic demonstration.

In [1]:
from pathlib import Path
import sys


def find_repo_root(start_path=None):
    """
    Find the PartnerLens repository root whether the notebook
    is started from the repository root or the notebooks folder.
    """
    current_path = Path(start_path or Path.cwd()).resolve()

    for candidate in [current_path, *current_path.parents]:
        required_items = [
            candidate / "src",
            candidate / "data",
            candidate / "configs",
            candidate / "notebooks",
        ]

        if all(item.exists() for item in required_items):
            return candidate

    raise FileNotFoundError(
        "PartnerLens repository root was not found. "
        "Start Jupyter from inside the partnerlens-copilot repository."
    )


REPO_ROOT = find_repo_root()

RAW_DIR = REPO_ROOT / "data" / "raw"
PROCESSED_DIR = REPO_ROOT / "data" / "processed"
OUTPUTS_DIR = REPO_ROOT / "outputs"
EVALUATION_DIR = REPO_ROOT / "evaluation"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repository root:", REPO_ROOT)
print("Raw data:", RAW_DIR)
print("Processed data:", PROCESSED_DIR)
print("Outputs:", OUTPUTS_DIR)
print("Evaluation:", EVALUATION_DIR)

Repository root: C:\Users\nafee\partnerlens-copilot
Raw data: C:\Users\nafee\partnerlens-copilot\data\raw
Processed data: C:\Users\nafee\partnerlens-copilot\data\processed
Outputs: C:\Users\nafee\partnerlens-copilot\outputs
Evaluation: C:\Users\nafee\partnerlens-copilot\evaluation


In [2]:
import os
import pandas as pd
import numpy as np

raw_files = {
    "partners": RAW_DIR / "partners.csv",
    "partner_pricing": RAW_DIR / "partner_pricing.csv",
    "monthly_partner_metrics": RAW_DIR / "monthly_partner_metrics.csv",
}

missing_files = [
    str(file_path)
    for file_path in raw_files.values()
    if not file_path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "The following raw PartnerLens files are missing:\n"
        + "\n".join(missing_files)
    )

partners = pd.read_csv(raw_files["partners"])
partner_pricing = pd.read_csv(raw_files["partner_pricing"])
monthly_partner_metrics = pd.read_csv(
    raw_files["monthly_partner_metrics"]
)

print("partners:", partners.shape)
print("partner_pricing:", partner_pricing.shape)
print(
    "monthly_partner_metrics:",
    monthly_partner_metrics.shape,
)

partners: (5000, 18)
partner_pricing: (5000, 14)
monthly_partner_metrics: (120000, 14)


Step 2: Inspect Columns

In [3]:
print("partners columns:")
print(partners.columns.tolist())

print("\npartner_pricing columns:")
print(partner_pricing.columns.tolist())

print("\nmonthly_partner_metrics columns:")
print(monthly_partner_metrics.columns.tolist())

partners columns:
['partner_id', 'partner_name', 'partner_type', 'industry_vertical', 'partner_size', 'partner_status', 'country', 'state', 'city', 'region', 'integration_type', 'sales_channel', 'risk_tier', 'kyc_status', 'pci_level', 'onboarding_date', 'relationship_manager_region', 'synthetic_record_flag']

partner_pricing columns:
['assignment_id', 'partner_id', 'pricing_plan_id', 'recommended_pricing_plan_id', 'effective_date', 'expiration_date', 'review_due_date', 'negotiated_bps', 'negotiated_per_txn_fee_usd', 'monthly_minimum_fee_usd', 'exception_flag', 'exception_type', 'approval_status', 'approved_by_role']

monthly_partner_metrics columns:
['partner_id', 'month', 'txn_count', 'payment_volume_usd', 'avg_ticket_usd', 'txn_growth_pct', 'volume_growth_pct', 'chargeback_rate', 'auth_approval_rate', 'active_merchants', 'refunds_usd', 'net_revenue_usd', 'gross_margin_usd', 'gross_margin_rate']


Step 3: Create Clean Working Copies

In [4]:
partners_clean = partners.copy()
partner_pricing_clean = partner_pricing.copy()
monthly_partner_metrics_clean = (
    monthly_partner_metrics.copy()
)

Step 4: Standardize Partner Master Columns

In [5]:
partners_clean["industry"] = partners_clean["industry_vertical"]
partners_clean["status"] = partners_clean["partner_status"]

partners_clean.head()

,partner_id,partner_name,partner_type,industry_vertical,partner_size,partner_status,country,state,city,region,integration_type,sales_channel,risk_tier,kyc_status,pci_level,onboarding_date,relationship_manager_region,synthetic_record_flag,industry,status
0,P000001,NorthStar Commerce 0001,Technology Platform,Travel,SMB,Suspended,US,NY,New York,Northeast,API,Direct,High,Approved,Level 3,2019-06-15,Northeast,1,Travel,Suspended
1,P000002,Ironwood Market 0002,ISV,SaaS,Mid-Market,Pilot,US,VA,Norfolk,South,SDK,Direct,Low,Approved,Level 4,2020-02-13,South,1,SaaS,Pilot
2,P000003,Evergreen Commerce 0003,Technology Platform,Professional Services,SMB,Active,US,CO,Colorado Springs,West,API,Direct,Low,Approved,Level 3,2023-03-12,West,1,Professional Services,Active
3,P000004,Summit Platform 0004,ISO/Reseller,Gaming,SMB,Active,US,GA,Augusta,South,API,Referral,Medium,Approved,Level 4,2020-02-13,South,1,Gaming,Active
4,P000005,Prairie PayTech 0005,Merchant Acquirer,Healthcare,SMB,Active,US,CA,San Diego,West,API,Direct,High,Approved,Level 4,2022-03-09,West,1,Healthcare,Active


Step 5: Standardize Pricing Columns

In [6]:
partner_pricing_clean["pricing_tier"] = partner_pricing_clean["pricing_plan_id"]

partner_pricing_clean["effective_date"] = pd.to_datetime(
    partner_pricing_clean["effective_date"],
    errors="coerce"
)

partner_pricing_clean["review_due_date"] = pd.to_datetime(
    partner_pricing_clean["review_due_date"],
    errors="coerce"
)

partner_pricing_clean["expiration_date"] = pd.to_datetime(
    partner_pricing_clean["expiration_date"],
    errors="coerce"
)

today = pd.Timestamp("2026-06-28")
next_90_days = today + pd.Timedelta(days=90)

partner_pricing_clean["contract_status"] = np.select(
    [
        partner_pricing_clean["approval_status"].eq("Expired"),
        partner_pricing_clean["review_due_date"] < today,
        partner_pricing_clean["review_due_date"].between(today, next_90_days)
    ],
    [
        "Expired",
        "Review Overdue",
        "Review Due Soon"
    ],
    default="Active"
)

partner_pricing_clean[
    [
        "assignment_id",
        "partner_id",
        "pricing_plan_id",
        "pricing_tier",
        "recommended_pricing_plan_id",
        "approval_status",
        "review_due_date",
        "contract_status"
    ]
].head()

,assignment_id,partner_id,pricing_plan_id,pricing_tier,recommended_pricing_plan_id,approval_status,review_due_date,contract_status
0,A000001,P000001,HIGH_RISK,HIGH_RISK,HIGH_RISK,Not Required,2025-04-11,Review Overdue
1,A000002,P000002,PLATFORM_ISV,PLATFORM_ISV,PLATFORM_ISV,Not Required,2025-02-06,Review Overdue
2,A000003,P000003,PLATFORM_ISV,PLATFORM_ISV,PLATFORM_ISV,Not Required,2026-01-04,Review Overdue
3,A000004,P000004,HIGH_RISK,HIGH_RISK,HIGH_RISK,Not Required,2027-02-07,Active
4,A000005,P000005,HIGH_RISK,HIGH_RISK,HIGH_RISK,Not Required,2027-08-27,Active


Step 6: Standardize Monthly Metrics Columns

In [7]:
monthly_partner_metrics_clean = monthly_partner_metrics_clean.reset_index(drop=True)

monthly_partner_metrics_clean["metric_id"] = [
    f"M{str(i + 1).zfill(6)}" for i in range(len(monthly_partner_metrics_clean))
]

monthly_partner_metrics_clean["transaction_month"] = pd.to_datetime(
    monthly_partner_metrics_clean["month"],
    errors="coerce"
)

monthly_partner_metrics_clean["transaction_count"] = monthly_partner_metrics_clean["txn_count"]
monthly_partner_metrics_clean["transaction_volume"] = monthly_partner_metrics_clean["payment_volume_usd"]
monthly_partner_metrics_clean["revenue"] = monthly_partner_metrics_clean["net_revenue_usd"]
monthly_partner_metrics_clean["growth_pct"] = monthly_partner_metrics_clean["txn_growth_pct"]

monthly_partner_metrics_clean[
    [
        "metric_id",
        "partner_id",
        "transaction_month",
        "transaction_count",
        "transaction_volume",
        "revenue",
        "growth_pct"
    ]
].head()

,metric_id,partner_id,transaction_month,transaction_count,transaction_volume,revenue,growth_pct
0,M000001,P000001,2024-01-01,100,10526.09,3121.44,NaN
1,M000002,P000001,2024-02-01,105,11242.13,3129.58,5.00
2,M000003,P000001,2024-03-01,108,12009.93,3138.08,2.86
3,M000004,P000001,2024-04-01,132,13850.19,3160.15,22.22
4,M000005,P000001,2024-05-01,122,14086.19,3161.65,-7.58


Step 7: Check Missing Values

In [8]:
print("Missing values in partners_clean:")
print(partners_clean.isnull().sum().sort_values(ascending=False))

print("\nMissing values in partner_pricing_clean:")
print(partner_pricing_clean.isnull().sum().sort_values(ascending=False))

print("\nMissing values in monthly_partner_metrics_clean:")
print(monthly_partner_metrics_clean.isnull().sum().sort_values(ascending=False))

Missing values in partners_clean:
partner_id                     0
partner_name                   0
partner_type                   0
industry_vertical              0
partner_size                   0
partner_status                 0
country                        0
state                          0
city                           0
region                         0
integration_type               0
sales_channel                  0
risk_tier                      0
kyc_status                     0
pci_level                      0
onboarding_date                0
relationship_manager_region    0
synthetic_record_flag          0
industry                       0
status                         0
dtype: int64

Missing values in partner_pricing_clean:
expiration_date                5000
approved_by_role               4170
exception_type                 4122
assignment_id                     0
recommended_pricing_plan_id       0
effective_date                    0
pricing_plan_id                   0

Step 8: Validate Primary Keys

In [9]:
duplicate_partner_ids = partners_clean["partner_id"].duplicated().sum()
duplicate_assignment_ids = partner_pricing_clean["assignment_id"].duplicated().sum()
duplicate_metric_ids = monthly_partner_metrics_clean["metric_id"].duplicated().sum()

duplicate_partner_months = monthly_partner_metrics_clean.duplicated(
    subset=["partner_id", "month"]
).sum()

print("Duplicate partner_id values:", duplicate_partner_ids)
print("Duplicate assignment_id values:", duplicate_assignment_ids)
print("Duplicate metric_id values:", duplicate_metric_ids)
print("Duplicate partner-month rows:", duplicate_partner_months)

Duplicate partner_id values: 0
Duplicate assignment_id values: 0
Duplicate metric_id values: 0
Duplicate partner-month rows: 0


Step 9: Validate Foreign Keys

In [10]:
pricing_unmatched = partner_pricing_clean[
    ~partner_pricing_clean["partner_id"].isin(partners_clean["partner_id"])
]

metrics_unmatched = monthly_partner_metrics_clean[
    ~monthly_partner_metrics_clean["partner_id"].isin(partners_clean["partner_id"])
]

print("Pricing records without matching partner:", len(pricing_unmatched))
print("Metric records without matching partner:", len(metrics_unmatched))

Pricing records without matching partner: 0
Metric records without matching partner: 0


Step 10: Check Category Values

In [11]:
print("Partner status values:")
print(partners_clean["partner_status"].value_counts())

print("\nIndustry vertical values:")
print(partners_clean["industry_vertical"].value_counts())

print("\nPartner type values:")
print(partners_clean["partner_type"].value_counts())

print("\nPartner size values:")
print(partners_clean["partner_size"].value_counts())

print("\nPricing tier values:")
print(partner_pricing_clean["pricing_tier"].value_counts())

print("\nApproval status values:")
print(partner_pricing_clean["approval_status"].value_counts())

print("\nDerived contract status values:")
print(partner_pricing_clean["contract_status"].value_counts())

Partner status values:
partner_status
Active        4013
Pilot          387
Dormant        292
Suspended      223
Terminated      85
Name: count, dtype: int64

Industry vertical values:
industry_vertical
Retail                   721
SaaS                     575
Fintech                  497
Healthcare               466
Logistics                415
Professional Services    405
Hospitality              378
Utilities                345
Travel                   309
Education                261
Nonprofit                254
Gaming                   218
Government               156
Name: count, dtype: int64

Partner type values:
partner_type
ISO/Reseller             927
Technology Platform      834
ISV                      766
Merchant Acquirer        655
Financial Institution    590
Marketplace              508
Gateway                  473
Bank Sponsor             247
Name: count, dtype: int64

Partner size values:
partner_size
SMB           1974
Mid-Market    1603
Enterprise    1006
Strategi

Step 11: Check Numeric Ranges

In [12]:
partner_pricing_clean[
    [
        "negotiated_bps",
        "negotiated_per_txn_fee_usd",
        "monthly_minimum_fee_usd"
    ]
].describe()

,negotiated_bps,negotiated_per_txn_fee_usd,monthly_minimum_fee_usd
count,5000.000000,5000.000000,5000.00000
mean,69.691442,0.063163,1378.41860
std,27.321828,0.023550,1471.19655
min,5.000000,0.013400,0.00000
25%,49.487500,0.045400,99.00000
50%,69.575000,0.059800,499.00000
75%,91.055000,0.081300,1999.00000
max,123.570000,0.113800,4999.00000


In [13]:
monthly_partner_metrics_clean[
    [
        "transaction_count",
        "transaction_volume",
        "avg_ticket_usd",
        "growth_pct",
        "volume_growth_pct",
        "chargeback_rate",
        "auth_approval_rate",
        "revenue",
        "gross_margin_usd",
        "gross_margin_rate"
    ]
].describe()

,transaction_count,transaction_volume,avg_ticket_usd,growth_pct,volume_growth_pct,chargeback_rate,auth_approval_rate,revenue,gross_margin_usd,gross_margin_rate
count,1.200000e+05,1.200000e+05,120000.000000,115000.000000,115000.000000,120000.000000,120000.000000,1.200000e+05,1.200000e+05,120000.000000
mean,4.881905e+04,4.011550e+06,121.867150,2.194633,1.922911,0.005826,0.915094,2.537831e+04,1.091384e+04,0.471528
std,1.792433e+05,1.092624e+07,76.398548,11.419830,8.761205,0.004911,0.034864,8.513043e+04,3.442730e+04,0.113436
min,1.000000e+01,1.000000e+03,11.220000,-41.310000,-31.330000,0.000100,0.764890,1.108000e+01,4.030000e+00,0.050000
25%,1.510000e+03,1.560190e+05,67.150000,-5.730000,-4.030000,0.002170,0.892400,1.986380e+03,9.232325e+02,0.406500
50%,5.749000e+03,5.785573e+05,97.560000,1.560000,1.540000,0.004690,0.917660,5.513815e+03,2.446735e+03,0.489900
75%,2.814300e+04,2.769361e+06,159.230000,9.390000,7.420000,0.008020,0.940100,1.801437e+04,8.354802e+03,0.553800
max,1.161502e+07,3.009404e+08,500.850000,65.350000,48.310000,0.027140,0.995000,4.627920e+06,2.419939e+06,0.750000


Step 12: Test Business Question Coverage

Question 1

Show me technology partners in Arizona with transaction growth above 20%.

In [14]:
q1_data = partners_clean.merge(
    monthly_partner_metrics_clean,
    on="partner_id",
    how="inner"
)

q1_result = q1_data[
    (q1_data["partner_type"].str.contains("Technology", case=False, na=False)) &
    (q1_data["state"] == "AZ") &
    (q1_data["growth_pct"] > 20)
]

q1_result[
    [
        "partner_id",
        "partner_name",
        "partner_type",
        "industry_vertical",
        "state",
        "transaction_month",
        "growth_pct"
    ]
].head(10)

,partner_id,partner_name,partner_type,industry_vertical,state,transaction_month,growth_pct
155,P000007,Clearwater Solutions 0007,Technology Platform,Retail,AZ,2024-12-01,27.44
166,P000007,Clearwater Solutions 0007,Technology Platform,Retail,AZ,2025-11-01,24.59
167,P000007,Clearwater Solutions 0007,Technology Platform,Retail,AZ,2025-12-01,31.87
4036,P000169,Brightpath PayWorks 0169,Technology Platform,Gaming,AZ,2024-05-01,32.07
4047,P000169,Brightpath PayWorks 0169,Technology Platform,Gaming,AZ,2025-04-01,26.28
9057,P000378,RedRock Solutions 0378,Technology Platform,Retail,AZ,2024-10-01,21.95
9069,P000378,RedRock Solutions 0378,Technology Platform,Retail,AZ,2025-10-01,34.85
10188,P000425,Summit Market 0425,Technology Platform,Education,AZ,2025-01-01,27.37
12380,P000516,Riverbend Solutions 0516,Technology Platform,Professional Services,AZ,2025-09-01,32.81
14388,P000600,Cedar Gateway 0600,Technology Platform,Healthcare,AZ,2025-01-01,23.59


In [15]:
print("Technology partners in AZ with growth > 20%:", len(q1_result))

Technology partners in AZ with growth > 20%: 85


Question 2

Which partners are on the Strategic pricing tier?

In [16]:
q2_data = partners_clean.merge(
    partner_pricing_clean,
    on="partner_id",
    how="inner"
)

q2_result = q2_data[q2_data["pricing_tier"] == "STRATEGIC"]

q2_result[
    [
        "partner_id",
        "partner_name",
        "pricing_tier",
        "recommended_pricing_plan_id",
        "approval_status"
    ]
].head(10)

,partner_id,partner_name,pricing_tier,recommended_pricing_plan_id,approval_status
20,P000021,Granite Gateway 0021,STRATEGIC,STRATEGIC,Not Required
27,P000028,NorthStar Merchant Services 0028,STRATEGIC,STRATEGIC,Not Required
42,P000043,Prairie PayTech 0043,STRATEGIC,STRATEGIC,Not Required
55,P000056,Cedar PayTech 0056,STRATEGIC,STRATEGIC,Not Required
65,P000066,Copper Financial 0066,STRATEGIC,STRATEGIC,Approved
66,P000067,Evergreen Processing 0067,STRATEGIC,STRATEGIC,Not Required
79,P000080,Granite Market 0080,STRATEGIC,STRATEGIC,Not Required
92,P000093,RedRock Payments 0093,STRATEGIC,STRATEGIC,Not Required
107,P000108,Copper Commerce 0108,STRATEGIC,STRATEGIC,Not Required
110,P000111,Maple Checkout 0111,STRATEGIC,STRATEGIC,Not Required


In [17]:
print("Strategic pricing partners:", len(q2_result))

Strategic pricing partners: 391


Question 3

Which partners have pricing reviews due in the next 90 days?

In [18]:
q3_data = partners_clean.merge(
    partner_pricing_clean,
    on="partner_id",
    how="inner"
)

q3_result = q3_data[
    q3_data["review_due_date"].between(today, next_90_days)
]

q3_result[
    [
        "partner_id",
        "partner_name",
        "pricing_tier",
        "approval_status",
        "review_due_date",
        "contract_status"
    ]
].head(10)

,partner_id,partner_name,pricing_tier,approval_status,review_due_date,contract_status
5,P000006,Brightpath PayWorks 0006,PLATFORM_ISV,Not Required,2026-09-08,Review Due Soon
11,P000012,Harbor PayWorks 0012,SCALE,Not Required,2026-08-12,Review Due Soon
28,P000029,Summit Financial 0029,HIGH_RISK,Not Required,2026-09-22,Review Due Soon
58,P000059,Sunrise Commerce 0059,GROWTH,Not Required,2026-08-23,Review Due Soon
65,P000066,Copper Financial 0066,STRATEGIC,Approved,2026-08-25,Review Due Soon
89,P000090,Maple Commerce 0090,HIGH_RISK,Not Required,2026-09-26,Review Due Soon
90,P000091,Ironwood Market 0091,LAUNCH,Not Required,2026-07-26,Review Due Soon
92,P000093,RedRock Payments 0093,STRATEGIC,Not Required,2026-07-17,Review Due Soon
93,P000094,Cedar PayWorks 0094,GROWTH,Not Required,2026-08-01,Review Due Soon
107,P000108,Copper Commerce 0108,STRATEGIC,Not Required,2026-08-15,Review Due Soon


In [19]:
print("Pricing reviews due in next 90 days:", len(q3_result))

Pricing reviews due in next 90 days: 361


Question 4

Which industry has the highest average transaction volume?

In [20]:
q4_data = partners_clean.merge(
    monthly_partner_metrics_clean,
    on="partner_id",
    how="inner"
)

q4_result = (
    q4_data
    .groupby("industry_vertical")["transaction_volume"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

q4_result

,industry_vertical,transaction_volume
0,Gaming,5.018721e+06
1,Travel,4.521112e+06
2,Fintech,4.449627e+06
3,Utilities,4.153576e+06
4,Logistics,4.149629e+06
5,Healthcare,3.949840e+06
6,Government,3.896428e+06
7,Hospitality,3.887503e+06
8,Retail,3.876467e+06
9,Professional Services,3.814276e+06


Question 5

Show the top 5 partners by revenue.

In [21]:
q5_data = partners_clean.merge(
    monthly_partner_metrics_clean,
    on="partner_id",
    how="inner"
)

q5_result = (
    q5_data
    .groupby(["partner_id", "partner_name"])["revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
)

q5_result

,partner_id,partner_name,revenue
0,P000610,Skyline Payments 0610,64847292.41
1,P002651,RedRock PayTech 2651,28539531.11
2,P003122,Cedar Market 3122,27485493.95
3,P004602,NorthStar Commerce 4602,26392278.10
4,P001422,Brightpath Platform 1422,25564051.57


Question 6

Which active partners are on pricing exceptions?

In [22]:
q6_data = partners_clean.merge(
    partner_pricing_clean,
    on="partner_id",
    how="inner"
)

q6_result = q6_data[
    (q6_data["partner_status"] == "Active") &
    (q6_data["exception_flag"] == 1)
]

q6_result[
    [
        "partner_id",
        "partner_name",
        "partner_status",
        "pricing_tier",
        "exception_flag",
        "exception_type",
        "negotiated_bps",
        "approval_status"
    ]
].head(10)

,partner_id,partner_name,partner_status,pricing_tier,exception_flag,exception_type,negotiated_bps,approval_status
19,P000020,Oakstone Gateway 0020,Active,PLATFORM_ISV,1,Discounted bps,29.94,Approved
23,P000024,RedRock PayTech 0024,Active,PLATFORM_ISV,1,Legacy plan extension,37.24,Pending
26,P000027,Prairie Financial 0027,Active,LEGACY_STANDARD,1,Discounted bps,59.12,Rejected
40,P000041,Cedar Merchant Services 0041,Active,LEGACY_STANDARD,1,High-risk waiver,72.08,Pending
43,P000044,Silverline Financial 0044,Active,LEGACY_STANDARD,1,Legacy plan extension,48.50,Approved
45,P000046,NorthStar Checkout 0046,Active,SCALE,1,High-risk waiver,47.83,Approved
50,P000051,Harbor Market 0051,Active,GROWTH,1,High-risk waiver,66.50,Approved
51,P000052,Riverbend Gateway 0052,Active,PROMO_GROWTH,1,Discounted bps,38.70,Expired
60,P000061,Skyline Platform 0061,Active,SCALE,1,Volume threshold waiver,30.06,Expired
61,P000062,Oakstone Financial 0062,Active,SCALE,1,Volume threshold waiver,49.63,Approved


In [23]:
print("Active partners with pricing exceptions:", len(q6_result))

Active partners with pricing exceptions: 681


Step 13: Create Phase 3 Validation Summary

In [24]:
validation_summary = pd.DataFrame({
    "Validation Check": [
        "partners_clean row count",
        "partner_pricing_clean row count",
        "monthly_partner_metrics_clean row count",
        "Duplicate partner_id in partners_clean",
        "Duplicate assignment_id in partner_pricing_clean",
        "Duplicate metric_id in monthly_partner_metrics_clean",
        "Duplicate partner-month records in monthly_partner_metrics_clean",
        "Pricing records without matching partner",
        "Metric records without matching partner",
        "Technology Platform partners in AZ with growth > 20%",
        "Partners on STRATEGIC pricing tier",
        "Pricing reviews due in next 90 days",
        "Active partners with pricing exceptions"
    ],
    "Result": [
        len(partners_clean),
        len(partner_pricing_clean),
        len(monthly_partner_metrics_clean),
        duplicate_partner_ids,
        duplicate_assignment_ids,
        duplicate_metric_ids,
        duplicate_partner_months,
        len(pricing_unmatched),
        len(metrics_unmatched),
        len(q1_result),
        len(q2_result),
        len(q3_result),
        len(q6_result)
    ]
})

validation_summary

,Validation Check,Result
0,partners_clean row count,5000
1,partner_pricing_clean row count,5000
2,monthly_partner_metrics_clean row count,120000
3,Duplicate partner_id in partners_clean,0
4,Duplicate assignment_id in partner_pricing_clean,0
5,Duplicate metric_id in monthly_partner_metrics...,0
6,Duplicate partner-month records in monthly_par...,0
7,Pricing records without matching partner,0
8,Metric records without matching partner,0
9,Technology Platform partners in AZ with growth...,85


Step 14: Save Cleaned Files for Phase 4

In [25]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

partners_output = (
    PROCESSED_DIR / "partner_master_clean.csv"
)
pricing_output = (
    PROCESSED_DIR / "partner_pricing_clean.csv"
)
metrics_output = (
    PROCESSED_DIR / "partner_metrics_clean.csv"
)
validation_output = (
    PROCESSED_DIR / "phase3_validation_summary.csv"
)

partners_clean.to_csv(
    partners_output,
    index=False,
)

partner_pricing_clean.to_csv(
    pricing_output,
    index=False,
)

monthly_partner_metrics_clean.to_csv(
    metrics_output,
    index=False,
)

validation_summary.to_csv(
    validation_output,
    index=False,
)

print("Saved:", partners_output)
print("Saved:", pricing_output)
print("Saved:", metrics_output)
print("Saved:", validation_output)

Saved: C:\Users\nafee\partnerlens-copilot\data\processed\partner_master_clean.csv
Saved: C:\Users\nafee\partnerlens-copilot\data\processed\partner_pricing_clean.csv
Saved: C:\Users\nafee\partnerlens-copilot\data\processed\partner_metrics_clean.csv
Saved: C:\Users\nafee\partnerlens-copilot\data\processed\phase3_validation_summary.csv


Step 15: Save Everything into One Excel Workbook

In [26]:
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [27]:
excel_output_path = (
    PROCESSED_DIR
    / "partnerlens_phase3_validated_dataset.xlsx"
)

with pd.ExcelWriter(
    excel_output_path,
    engine="openpyxl",
) as writer:
    partners_clean.to_excel(
        writer,
        sheet_name="partners",
        index=False,
    )

    partner_pricing_clean.to_excel(
        writer,
        sheet_name="partner_pricing",
        index=False,
    )

    monthly_partner_metrics_clean.to_excel(
        writer,
        sheet_name="monthly_metrics",
        index=False,
    )

    validation_summary.to_excel(
        writer,
        sheet_name="validation_summary",
        index=False,
    )

    q1_result.head(100).to_excel(
        writer,
        sheet_name="q1_tech_az_growth",
        index=False,
    )

    q2_result.head(100).to_excel(
        writer,
        sheet_name="q2_strategic_pricing",
        index=False,
    )

    q3_result.head(100).to_excel(
        writer,
        sheet_name="q3_review_due",
        index=False,
    )

    q4_result.to_excel(
        writer,
        sheet_name="q4_volume_industry",
        index=False,
    )

    q5_result.to_excel(
        writer,
        sheet_name="q5_top_revenue",
        index=False,
    )

print("Workbook saved to:", excel_output_path)

Workbook saved to: C:\Users\nafee\partnerlens-copilot\data\processed\partnerlens_phase3_validated_dataset.xlsx


In [28]:
from src.data_preparation import (
    print_validation_summary,
    validate_processed_data,
)

phase8_validation_result = validate_processed_data(
    processed_dir=PROCESSED_DIR,
    fail_on_error=False,
)

print_validation_summary(
    phase8_validation_result
)

if not phase8_validation_result.passed:
    raise ValueError(
        "Phase 8 data validation failed. "
        "Review artifacts/phase8/data_validation_report.csv."
    )


PartnerLens Phase 8 Data Validation
-----------------------------------
Overall status: PASSED
Files inspected: 5
Passed checks: 86
Warnings: 0
Failed checks: 0
Validation report: C:\Users\nafee\partnerlens-copilot\artifacts\phase8\data_validation_report.csv
Data manifest: C:\Users\nafee\partnerlens-copilot\artifacts\phase8\data_manifest.csv


In [29]:
required_processed_files = [
    PROCESSED_DIR / "partner_master_clean.csv",
    PROCESSED_DIR / "partner_pricing_clean.csv",
    PROCESSED_DIR / "partner_metrics_clean.csv",
    PROCESSED_DIR / "partner_current_preview_1000.csv",
    PROCESSED_DIR / "phase3_validation_summary.csv",
]

file_verification = pd.DataFrame(
    [
        {
            "file": file_path.name,
            "exists": file_path.exists(),
            "size_bytes": (
                file_path.stat().st_size
                if file_path.exists()
                else None
            ),
        }
        for file_path in required_processed_files
    ]
)

file_verification

,file,exists,size_bytes
0,partner_master_clean.csv,True,844394
1,partner_pricing_clean.csv,True,605072
2,partner_metrics_clean.csv,True,18070358
3,partner_current_preview_1000.csv,True,478467
4,phase3_validation_summary.csv,True,630
